**MathNet Category Deep Dive**

This notebook summarizes the extra analysis requested after the long-joint run.

**Overall Long-Joint Best Metrics**

- Accuracy: 0.8590
- Balanced accuracy: 0.8487
- Macro F1: 0.8485
- Weighted F1: 0.8594
- Log loss: 0.4300
- Classification perplexity: 1.5372
- Macro ROC AUC: 0.9631
- Macro PR AUC: 0.8975
- ECE with 15 bins: 0.0358

**Checkpoint Comparison**

| checkpoint | accuracy | loss | macro_f1 |
| --- | --- | --- | --- |
| supervised_best | 0.8568 | 0.4185 | 0.8475 |
| finetune_best | 0.8535 | 0.4225 | 0.8439 |
| joint_best | 0.8594 | 0.4250 | 0.8490 |
| long_joint_best | 0.8590 | 0.4331 | 0.8485 |

**Per-Class Metrics**

| category | support | predicted_count | correct | accuracy_recall | precision | f1 | roc_auc_ovr | pr_auc_ovr | brier_ovr | mean_ce_loss | classification_perplexity | mean_true_probability | mean_confidence | mean_entropy | mean_margin |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Algebra | 786 | 769 | 657 | 0.8359 | 0.8544 | 0.8450 | 0.9550 | 0.9124 | 0.0670 | 0.4971 | 1.6439 | 0.7850 | 0.8859 | 0.3591 | 0.8018 |
| Combinatorics | 587 | 640 | 489 | 0.8330 | 0.7641 | 0.7971 | 0.9485 | 0.8265 | 0.0708 | 0.5323 | 1.7028 | 0.7641 | 0.8622 | 0.3985 | 0.7624 |
| Geometry | 832 | 825 | 793 | 0.9531 | 0.9612 | 0.9572 | 0.9924 | 0.9831 | 0.0217 | 0.1610 | 1.1747 | 0.9179 | 0.9449 | 0.2116 | 0.9088 |
| Number Theory | 519 | 490 | 401 | 0.7726 | 0.8184 | 0.7948 | 0.9566 | 0.8679 | 0.0576 | 0.6438 | 1.9037 | 0.7200 | 0.8524 | 0.4320 | 0.7429 |

**Feature Stats**

| category | n | median_chars | median_word_tokens | median_bpe_tokens | truncated_rate | image_metadata_rate | problem_markdown_image_rate | geometry_cue_rate | combinatorics_cue_rate | algebra_cue_rate | number_theory_cue_rate |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Algebra | 786 | 210.5000 | 66.0000 | 89.0000 | 0.0267 | 0.0204 | 0.0076 | 0.0293 | 0.1578 | 0.5445 | 0.2621 |
| Combinatorics | 587 | 359.0000 | 80.0000 | 114.0000 | 0.0886 | 0.1533 | 0.0443 | 0.2385 | 0.4889 | 0.1925 | 0.2998 |
| Geometry | 832 | 307.0000 | 83.0000 | 103.5000 | 0.0240 | 0.5000 | 0.0757 | 0.8089 | 0.1010 | 0.0553 | 0.0589 |
| Number Theory | 519 | 186.0000 | 50.0000 | 64.0000 | 0.0096 | 0.0135 | 0.0000 | 0.0809 | 0.1310 | 0.2582 | 0.6879 |

**Distinctive Training Words**

- Algebra: convergent, poenostavi, invertible, commutative, funktionen, funciones, funkcija, continuous, roots, realna, functies, ecuaciones, realni, composition, derivative
- Combinatorics: cities, flights, loses, colorings, flight, colonne, roads, tournament, making, vertically, dominoes, colouring, caselle, knight, chess
- Geometry: circumcircle, circumcenter, tangent, incircle, incenter, orthocenter, bisector, altitude, perp, altitudes, circumcircles, acb, bac, bisectors, widehat
- Number Theory: lcm, primzahl, divisors, premiers, divisori, deler, parfait, divisores, popoln, january, deljiva, kgv, diviseurs, primes, overbrace

**Interpretation**

On the current long-joint best checkpoint, Combinatorics does not beat Algebra by accuracy or F1. Algebra recall is 0.8359 and Combinatorics recall is 0.8330. Algebra F1 is 0.8450 and Combinatorics F1 is 0.7971.

**Why The Scores Look Discrepant**

There are two separate effects. First, different metrics answer different questions. Per-category accuracy in this report is the same thing as recall for that true class. It asks: of the examples that truly belong to this category, how many were recovered. Precision asks: of the examples predicted as this category, how many were actually correct. F1 combines precision and recall.

Combinatorics looks competitive on recall, but weak on precision. The model predicted Combinatorics 640 times, while the test set contains only 587 true Combinatorics examples. That means the model overuses the Combinatorics label. It correctly recovered 489 Combinatorics examples, but it also pulled in 151 false positives from other categories. This gives recall 0.8330 but precision only 0.7641, so F1 falls to 0.7971.

Algebra has almost the same recall, 0.8359, but much better precision, 0.8544. The model predicted Algebra 769 times for 786 true Algebra examples. It missed some Algebra examples, especially ones with grids, colors, chairs, socks, and other discrete story language, but when it did predict Algebra it was more often right. That is why Algebra F1 is 0.8450, much higher than Combinatorics F1.

Second, checkpoint choice changes small per-class recall comparisons. In the earlier short joint checkpoint, Combinatorics recall was 0.8382 and Algebra recall was 0.8295, so Combinatorics looked slightly better by recall. In the long-joint best checkpoint, Algebra recall is 0.8359 and Combinatorics recall is 0.8330, so Algebra is slightly better. The recall gap is tiny in both cases. The stable finding is not that Combinatorics is truly better than Algebra. The stable finding is that Geometry is much easier, and Number Theory is consistently harder.

AUC adds another wrinkle. AUC measures ranking quality for one category against the rest across all possible thresholds, not the quality of the final argmax label. Number Theory has ROC AUC 0.9566, which is strong, even though its argmax recall is only 0.7726. That means the model often assigns useful relative probability to Number Theory, but the final top label is frequently stolen by Algebra or Combinatorics. For the actual classifier, the confusion matrix and F1 are more directly meaningful than AUC alone.

Log loss and classification perplexity explain another difference. Number Theory has the worst mean cross-entropy loss, 0.6438, and worst classification perplexity, 1.9037. Combinatorics is next worst at 0.5323 loss and 1.7028 perplexity. Algebra is better at 0.4971 loss and 1.6439 perplexity. Geometry is far better at 0.1610 loss and 1.1747 perplexity. This tells us the model is not merely making more Number Theory mistakes. It is also less confident in the correct Number Theory label on average.

Combinatorics can look strong on recall because many examples contain concrete discrete-object cues like colorings, graphs, games, grids, chess, roads, cities, flights, and arrangements. The tradeoff is precision. The model overpredicts Combinatorics for 64 Algebra examples, 26 Geometry examples, and 61 Number Theory examples.

Number Theory does worse than Algebra because it is smaller, overlaps heavily with Algebra syntax, and often uses generic variables, equations, integer constraints, sequences, and polynomial-looking expressions. Its strong cues like prime, divisor, gcd, lcm, modulo, and congruent are helpful when present, but many Number Theory problems do not expose enough of those cues in the problem statement alone.

Geometry remains the easiest category because its vocabulary is highly separable. The test split geometry-cue rate is 0.804, far higher than Algebra or Number Theory.

**Graphs**

![model_comparison.png](model_comparison.png)
![per_category_scores.png](per_category_scores.png)
![confusion_matrix.png](confusion_matrix.png)
![loss_ppl_confidence.png](loss_ppl_confidence.png)
![confidence_histogram.png](confidence_histogram.png)
![calibration_curve.png](calibration_curve.png)
![roc_curves.png](roc_curves.png)
![pr_curves.png](pr_curves.png)
![cue_rates.png](cue_rates.png)
![text_length_and_markup.png](text_length_and_markup.png)
![long_joint_training_curve.png](long_joint_training_curve.png)


In [ ]:
from pathlib import Path
import json

metrics = json.loads(Path("deep_dive_metrics.json").read_text())
metrics["overall"]
